# pymefpp_run

This is a wrapper script to the mefpp engine to reduce the data that has been partioned and associated with the scripts

*pymefpp_partition_calibration
*pymefpp_partition_science
*pymefpp_associate

After running those scripts we end up with a  directory named reduced that contains the following tree structure:

* reduced/calibs
        * /bpm
        * /bias
        * /dome
        * /sky
    * /science
    
Below science we have all the science partition to be reduced.

This script will:

* take a science partition at a time
* execute the reduction cascade

The idea is to identify all biases, dome flats, sky flats that are needed for the full reduction of that science partitions, and only these, and reduce them. This permits a piecemeal approach to the work.


        
 

In [ ]:
from os import listdir
from os.path import isfile, join
import glob
from astropy.io import fits
import math
import operator
import copy
import os
import shutil
from datetime import datetime
from pprint import pprint

# here we import the local mefpp_run library
import sys
sys.path.insert(0, os.getcwd())
from mefpp_associate_reduction import *


In [ ]:
# we set the data reduction directories -- LONG TIME TO RUN
# 
root_dir = ""
raw_data_dir = ""
reduced_cal_dir = root_dir+'calibs/'
reduced_science_dir = root_dir+'science/'
log_dir = root_dir+'logs/'

# where to find mefpp module
mefpp_dir = "/home/lientur/Investigacion/OmegaCam/mefpp/"

os.chdir(root_dir)
global cwd
cwd = os.getcwd()
print('cwd: ', cwd)

# depending on size of data set this line might be very slow
science_list = glob.glob(reduced_science_dir+'**/science.list', recursive=True)
print('number of science partitions: ', len(science_list))


In [ ]:
# associations for the science frame 
sci_partitions = []
for science_file in science_list:
    sci_partitions.append(ExposuresPartition(science_file, 'SCIENCE'))


In [ ]:
# select data to reduce
print('--------------------- science frame associations -------------------')
science_partition_list = sci_partitions
i = 0
for partition in science_partition_list:
    print(i, partition.base_path)
    i+=1

In [ ]:
# we do the association here
science_partition_list = sci_partitions
for scipart in science_partition_list:

    bias_part_list = []
    dome_part_list = []
    sky_part_list = []
    
    sci_part_list = []

    # -------------- DATA ASSOCIATION SECTION ---------------
    print('scipart: '+scipart.base_path)
    sci_part_list.append(scipart)
    
    associateBPM(scipart, scipart)
    print('        bpm part:  '+scipart.bpmPartition.base_path)
    associateBias(scipart, scipart)
    print('        bias part:  '+scipart.biasPartition.base_path)
    bias_part_list.append(scipart.biasPartition)
    associateMFF(scipart, scipart)
    print('        MFF  part:  '+scipart.mffPartition.base_path)

    # associations for the MFF frame
    mffpart = scipart.mffPartition
    print('         mffpart: '+mffpart.base_path)
    associateBPM(mffpart, scipart)
    print('                  bpm  part:  '+mffpart.bpmPartition.base_path)
    associateBias(mffpart, scipart)
    print('                  bias part:  '+mffpart.biasPartition.base_path)
    bias_part_list.append(mffpart.biasPartition)
    associateDome(mffpart, scipart)
    print('                  dome part:  '+mffpart.domePartition.base_path)
    dome_part_list.append(mffpart.domePartition)
    associateSky(mffpart, scipart)
    print('                   sky part:  '+mffpart.skyPartition.base_path)
    sky_part_list.append(mffpart.skyPartition)

    # associations for the SKY frame
    skypart = mffpart.skyPartition
    print('  ---> skypart: '+skypart.base_path)
    associateBPM(skypart, scipart)
    print('                  bpm  part:  '+skypart.bpmPartition.base_path)
    associateBias(skypart, scipart)
    print('                  bias part:  '+skypart.biasPartition.base_path)
    bias_part_list.append(skypart.biasPartition)
    associateDome(skypart, scipart)
    print('                  dome part:  '+skypart.domePartition.base_path)
    dome_part_list.append(skypart.domePartition)

    # associations for the dome frame
    domepart = ExposuresPartition(skypart.domePartition.frame, 'FLAT,DOME')
    print('  ---> domepart: '+domepart.base_path)
    associateBPM(domepart, scipart)
    print('                  bpm  part:  '+domepart.bpmPartition.base_path)
    associateBias(domepart, scipart)
    print('                  bias part:  '+domepart.biasPartition.base_path)
    bias_part_list.append(domepart.biasPartition)
    print ('%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%')
    
    # --------------------- DATA REDUCTION SECTION ---------------------
    # here we reduce the bpm frames which for now is only a cp operation
    mefpp_reduce_bpm(mefpp_dir, reduced_cal_dir)

    # we start reducing the biases
    mefpp_reduce_bias(mefpp_dir, bias_part_list)

    # reduction of dome frames
    mefpp_reduce_dome(mefpp_dir, dome_part_list)

    # reduction of sky frames
    mefpp_reduce_sky(mefpp_dir, sky_part_list)
  
    # we reduce the science frames here
    mefpp_reduce_science(mefpp_dir, sci_part_list)
    
    # finally we normalize the image to set the units in e-/s.
    # light correction is applied if the filter are u,g,r o i_sdss
    mefpp_normalize_science(sci_part_list)
